<a href="https://colab.research.google.com/github/Tejaswi37/RL/blob/main/RL_ASSIGNMENT_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np
import pandas as pd

# Grid World:
# S0  S1
# WALL G

gamma = 0.9
theta = 1e-6

states = ["S0", "S1", "G"]
non_terminal_states = ["S0", "S1"]
actions = ["Up", "Down", "Left", "Right"]

rewards = {
    "normal": -1,
    "wall": -1,
    "goal": 10
}

grid = {
    (0, 0): "S0",
    (0, 1): "S1",
    (1, 0): "WALL",
    (1, 1): "G"
}

positions = {
    "S0": (0, 0),
    "S1": (0, 1),
    "G": (1, 1)
}

moves = {
    "Up": (-1, 0),
    "Down": (1, 0),
    "Left": (0, -1),
    "Right": (0, 1)
}

print("DETERMINISTIC GRID WORLD")
print("------------------------")
print("Grid:")
print("S0   S1")
print("#    G")
print("\nStates:", states)
print("Non-terminal states:", non_terminal_states)
print("Actions:", actions)
print("Discount factor gamma:", gamma)
print("Rewards:", rewards)
print("Convergence threshold:", theta)


DETERMINISTIC GRID WORLD
------------------------
Grid:
S0   S1
#    G

States: ['S0', 'S1', 'G']
Non-terminal states: ['S0', 'S1']
Actions: ['Up', 'Down', 'Left', 'Right']
Discount factor gamma: 0.9
Rewards: {'normal': -1, 'wall': -1, 'goal': 10}
Convergence threshold: 1e-06


In [ ]:

def transition(state, action):
    # Goal is terminal
    if state == "G":
        return "G", 0

    row, col = positions[state]
    dr, dc = moves[action]

    new_position = (row + dr, col + dc)

    # Hitting the boundary
    if new_position not in grid:
        return state, rewards["wall"]

    next_state = grid[new_position]

    # Hitting the wall
    if next_state == "WALL":
        return state, rewards["wall"]

    # Reaching the goal
    if next_state == "G":
        return "G", rewards["goal"]

    # Moving to a normal cell
    return next_state, rewards["normal"]


In [ ]:

transition_data = []

for state in non_terminal_states:
    for action in actions:
        next_state, reward = transition(state, action)

        transition_data.append({
            "Current State": state,
            "Action": action,
            "Next State": next_state,
            "Immediate Reward": reward
        })

transition_df = pd.DataFrame(transition_data)

print("TRANSITION MAPPING")
display(transition_df)

expected_pairs = len(non_terminal_states) * len(actions)

print("Expected state-action pairs:", expected_pairs)
print("Defined transitions:", len(transition_df))
print("All transitions defined:",
      len(transition_df) == expected_pairs)


TRANSITION MAPPING


,Current State,Action,Next State,Immediate Reward
0,S0,Up,S0,-1
1,S0,Down,S0,-1
2,S0,Left,S0,-1
3,S0,Right,S1,-1
4,S1,Up,S1,-1
5,S1,Down,G,10
6,S1,Left,S0,-1
7,S1,Right,S1,-1


Expected state-action pairs: 8
Defined transitions: 8
All transitions defined: True


In [ ]:

V = {
    "S0": 0.0,
    "S1": 0.0,
    "G": 0.0
}

print("INITIAL STATE VALUES")
for state, value in V.items():
    print(f"V({state}) = {value:.4f}")


INITIAL STATE VALUES
V(S0) = 0.0000
V(S1) = 0.0000
V(G) = 0.0000


In [ ]:

iteration = 0
history = []

while True:
    iteration += 1

    # Copy previous values
    old_V = V.copy()

    # Calculate all new values using old_V only
    new_V = old_V.copy()

    for state in non_terminal_states:
        q_values = []

        for action in actions:
            next_state, reward = transition(state, action)

            q = reward + gamma * old_V[next_state]
            q_values.append(q)

        new_V[state] = max(q_values)

    # Terminal state always has value zero
    new_V["G"] = 0.0

    # Maximum change in state values
    delta = max(
        abs(new_V[s] - old_V[s])
        for s in non_terminal_states
    )

    V = new_V

    history.append({
        "Iteration": iteration,
        "V(S0)": V["S0"],
        "V(S1)": V["S1"],
        "V(G)": V["G"],
        "Delta": delta
    })

    print(
        f"Iteration {iteration:2d}: "
        f"V(S0)={V['S0']:.6f}, "
        f"V(S1)={V['S1']:.6f}, "
        f"V(G)={V['G']:.6f}, "
        f"Delta={delta:.6f}"
    )

    if delta < theta:
        break

print("\nValue Iteration converged!")


Iteration  1: V(S0)=-1.000000, V(S1)=10.000000, V(G)=0.000000, Delta=10.000000
Iteration  2: V(S0)=8.000000, V(S1)=10.000000, V(G)=0.000000, Delta=9.000000
Iteration  3: V(S0)=8.000000, V(S1)=10.000000, V(G)=0.000000, Delta=0.000000

Value Iteration converged!


In [ ]:

history_df = pd.DataFrame(history)

print("ITERATION-WISE STATE VALUES")
display(history_df.round(6))

print("\nFINAL OPTIMAL STATE VALUES")
for state in states:
    print(f"V*({state}) = {V[state]:.6f}")


ITERATION-WISE STATE VALUES


,Iteration,V(S0),V(S1),V(G),Delta
0,1,-1.0,10.0,0.0,10.0
1,2,8.0,10.0,0.0,9.0
2,3,8.0,10.0,0.0,0.0



FINAL OPTIMAL STATE VALUES
V*(S0) = 8.000000
V*(S1) = 10.000000
V*(G) = 0.000000


In [ ]:

Q = {}

q_rows = []

for state in non_terminal_states:
    Q[state] = {}

    for action in actions:
        next_state, reward = transition(state, action)

        q_value = reward + gamma * V[next_state]
        Q[state][action] = q_value

        q_rows.append({
            "State": state,
            "Action": action,
            "Next State": next_state,
            "Reward": reward,
            "Q-value": q_value
        })

q_df = pd.DataFrame(q_rows)

print("OPTIMAL ACTION-VALUE TABLE")
display(q_df.round(6))


OPTIMAL ACTION-VALUE TABLE


,State,Action,Next State,Reward,Q-value
0,S0,Up,S0,-1,6.2
1,S0,Down,S0,-1,6.2
2,S0,Left,S0,-1,6.2
3,S0,Right,S1,-1,8.0
4,S1,Up,S1,-1,8.0
5,S1,Down,G,10,10.0
6,S1,Left,S0,-1,6.2
7,S1,Right,S1,-1,8.0


In [ ]:

optimal_policy = {}

print("OPTIMAL POLICY")

for state in non_terminal_states:
    max_q = max(Q[state].values())

    best_actions = [
        action for action, value in Q[state].items()
        if np.isclose(value, max_q, atol=theta)
    ]

    optimal_policy[state] = best_actions

    print(
        f"State {state}: "
        f"Optimal action(s) = {best_actions}, "
        f"Value = {max_q:.6f}"
    )


OPTIMAL POLICY
State S0: Optimal action(s) = ['Right'], Value = 8.000000
State S1: Optimal action(s) = ['Down'], Value = 10.000000


In [9]:

print("BELLMAN OPTIMALITY VERIFICATION")

verification_rows = []

for state in non_terminal_states:
    max_q = max(Q[state].values())
    state_value = V[state]

    verified = np.isclose(
        state_value, max_q, atol=theta
    )

    verification_rows.append({
        "State": state,
        "V*(s)": state_value,
        "Max Q*(s,a)": max_q,
        "Verified": verified
    })

verification_df = pd.DataFrame(verification_rows)
display(verification_df.round(6))

if verification_df["Verified"].all():
    print("SUCCESS: Bellman Optimality equation verified.")
else:
    print("Verification failed.")


BELLMAN OPTIMALITY VERIFICATION


,State,V*(s),"Max Q*(s,a)",Verified
0,S0,8.0,8.0,True
1,S1,10.0,10.0,True


SUCCESS: Bellman Optimality equation verified.


In [10]:

policy_symbols = {
    "Up": "↑",
    "Down": "↓",
    "Left": "←",
    "Right": "→"
}

print("OPTIMAL POLICY GRID")
print("-------------------")

for state in ["S0", "S1"]:
    symbols = [
        policy_symbols[action]
        for action in optimal_policy[state]
    ]
    print(f"{state}: {' / '.join(symbols)}")

print("WALL: #")
print("GOAL: G")


OPTIMAL POLICY GRID
-------------------
S0: →
S1: ↓
WALL: #
GOAL: G
